### **`When you don't have uniform primary GTV names in your dataset`**

---

In [ ]:
import os
import glob
import pandas as pd
import pydicom

# Backward compatibility for dcmrtstruct2nii
if not hasattr(pydicom, "read_file"):
    pydicom.read_file = pydicom.dcmread

from dcmrtstruct2nii import dcmrtstruct2nii, list_rt_structs

# === CONFIGURATION ===
base_dir = "./FOLLOW-UP_PATIENTS_FOR_2_YEARS_LRR__&__NON-LRR"
temporary_folder = "./convert_nifti_format"
output_folder = "./output"
GTV_MAPPING_FILE = "/home/radiomicsserver/Downloads/hasan/patient_report_20251108_145930.csv"

os.makedirs(temporary_folder, exist_ok=True)
os.makedirs(output_folder, exist_ok=True)

# === LOAD GTV MAPPING ===
print("Loading GTV mapping...")
gtv_df = pd.read_csv(GTV_MAPPING_FILE)
gtv_mapping = dict(zip(
    gtv_df["PatID"].astype(str).str.strip(), 
    gtv_df["Current name of GTV_Primary"].astype(str).str.strip()
))
print(f"✅ Loaded mapping for {len(gtv_mapping)} patients\n")

# === HELPER FUNCTIONS ===
def find_rtstruct_and_ct(patient_folder):
    subdirs = [d for d in os.listdir(patient_folder) if os.path.isdir(os.path.join(patient_folder, d))]
    subdirs_lower = [d.lower() for d in subdirs]

    rtstruct_folder = None
    ct_folder = None

    if "rtstruct" in subdirs_lower:
        rtstruct_folder = os.path.join(patient_folder, subdirs[subdirs_lower.index("rtstruct")])
    if "ct" in subdirs_lower:
        ct_folder = os.path.join(patient_folder, subdirs[subdirs_lower.index("ct")])

    rtstruct_file = None
    if rtstruct_folder and os.path.exists(rtstruct_folder):
        dcm_files = glob.glob(os.path.join(rtstruct_folder, "*.dcm"))
        if dcm_files:
            rtstruct_file = dcm_files[0]

    return rtstruct_file, ct_folder

# === MAIN PROCESSING ===
patient_folders = sorted(os.listdir(base_dir))
successful = 0
failed = 0
summary_report = []

for i, patient_id in enumerate(patient_folders, 1):
    patient_path = os.path.join(base_dir, patient_id)
    if not os.path.isdir(patient_path):
        continue

    print(f"[{i}/{len(patient_folders)}] {patient_id}")
    
    # Check if patient in mapping
    if patient_id not in gtv_mapping:
        print(f"  ⚠️  Not in mapping file")
        failed += 1
        summary_report.append({"PatientID": patient_id, "Status": "Failed", "Reason": "Not in mapping"})
        continue

    gtv_name = gtv_mapping[patient_id]
    
    # Find RTSTRUCT and CT
    rtstruct_path, ct_path = find_rtstruct_and_ct(patient_path)
    if not rtstruct_path or not ct_path:
        print(f"  ⚠️  Missing RTSTRUCT or CT")
        failed += 1
        summary_report.append({"PatientID": patient_id, "Status": "Failed", "Reason": "Missing files"})
        continue

    # Check if GTV exists
    try:
        structures = list_rt_structs(rtstruct_path)
    except Exception as e:
        print(f"  ⚠️  Error reading structures: {e}")
        failed += 1
        summary_report.append({"PatientID": patient_id, "Status": "Failed", "Reason": str(e)})
        continue

    if gtv_name not in structures:
        print(f"  ⚠️  GTV '{gtv_name}' not found. Available: {structures}")
        failed += 1
        summary_report.append({"PatientID": patient_id, "Status": "Failed", "Reason": f"GTV not found", "Available": str(structures)})
        continue

    print(f"  🩻 Using GTV: '{gtv_name}'")

    # Convert to NIFTI
    output_patient_folder = os.path.join(temporary_folder, patient_id)
    os.makedirs(output_patient_folder, exist_ok=True)

    try:
        dcmrtstruct2nii(
            rtstruct_path,
            ct_path,
            output_patient_folder,
            convert_original_dicom=True,
            structures=[gtv_name]
        )
        print(f"  ✅ Converted")
    except Exception as e:
        print(f"  ❌ Conversion failed: {e}")
        failed += 1
        summary_report.append({"PatientID": patient_id, "Status": "Failed", "Reason": str(e)})
        continue
    
    successful += 1
    summary_report.append({"PatientID": patient_id, "Status": "Success", "GTV_Used": gtv_name})

# Save summary
pd.DataFrame(summary_report).to_csv("conversion_summary.csv", index=False)

print(f"\n✅ Success: {successful} | ❌ Failed: {failed}")


🔄 CONVERTING CMC DICOM TO NIFTI

[ 1/30] Processing: HN00020_20240712_Prospective
     🩻 Using structure: 'gtv-1'
     ✅ Converted to NIFTI
[ 2/30] Processing: HN00021_20240712_Prospective
     🩻 Using structure: 'gtv-1'
     ✅ Converted to NIFTI
[ 3/30] Processing: HN00024_20240712_Prospective
     🩻 Using structure: 'gtv-1'
     ✅ Converted to NIFTI
[ 4/30] Processing: HN00028_20240712_Prospective
     🩻 Using structure: 'gtv-1'
     ✅ Converted to NIFTI
[ 5/30] Processing: HN00035_20240712_Prospective
     🩻 Using structure: 'gtv-1'
     ✅ Converted to NIFTI
[ 6/30] Processing: HN00039_20240712_Prospective
     🩻 Using structure: 'gtv-1'
     ✅ Converted to NIFTI
[ 7/30] Processing: HN00040_20240712_Prospective
     🩻 Using structure: 'gtv-1'
     ✅ Converted to NIFTI
[ 8/30] Processing: HN00056_20240712_Prospective
     🩻 Using structure: 'gtv-1'
     ✅ Converted to NIFTI
[ 9/30] Processing: HN00059_20240712_Prospective
     🩻 Using structure: 'gtv-1'
     ✅ Converted to NIFTI
[10

---